In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import randint
import warnings
warnings.filterwarnings('ignore')

In [2]:
housing = fetch_california_housing(as_frame=True)
X = housing.data  # Признаки (DataFrame)
y = housing.target  # Целевая переменная (Series)

X.shape

(20640, 8)

In [3]:
X.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42, 
    shuffle=True
)

print(X_train.shape[0])
print(X_test.shape[0])

16512
4128


In [5]:
# Создаём и обучаем модель KNN с произвольно выбранным K=5
k_initial = 5
knn_initial = KNeighborsRegressor(n_neighbors=k_initial)
knn_initial.fit(X_train, y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [6]:
# Делаем предсказания на тестовой выборке
y_pred_initial = knn_initial.predict(X_test)

In [7]:
# Оцениваем качество модели
mse_initial = mean_squared_error(y_test, y_pred_initial)
mae_initial = mean_absolute_error(y_test, y_pred_initial)
r2_initial = r2_score(y_test, y_pred_initial)

print(f"Среднеквадратичная ошибка (MSE):  {mse_initial:.4f}")
print(f"Средняя абсолютная ошибка (MAE): {mae_initial:.4f}")
print(f"Коэффициент детерминации (R²):   {r2_initial:.4f}")

  Среднеквадратичная ошибка (MSE):  1.1187
  Средняя абсолютная ошибка (MAE): 0.8128
  Коэффициент детерминации (R²):   0.1463


In [8]:
# --- ОПТИМИЗАЦИЯ ЧЕРЕЗ GridSearchCV ---
# Задаём сетку возможных значений K для перебора
param_grid = {'n_neighbors': list(range(1, 31))}  # K от 1 до 30

# Создаём модель KNN
knn = KNeighborsRegressor()

In [10]:
# Определяем стратегии кросс-валидации
# 1. K-Fold с 5 фолдами
kf5 = KFold(n_splits=5, shuffle=True, random_state=42)

# Проводим GridSearchCV с 5-фолдовой кросс-валидацией
grid_search_5 = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=kf5,
    scoring='neg_mean_squared_error',  # Используем отрицательную MSE (чем больше, тем лучше)
    n_jobs=-1,  # Используем все доступные ядра процессора
    verbose=1
)
grid_search_5.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,estimator,KNeighborsRegressor()
,param_grid,"{'n_neighbors': [1, 2, ...]}"
,scoring,'neg_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,8


In [11]:
# 2. K-Fold с 10 фолдами (вторая стратегия)
kf10 = KFold(n_splits=10, shuffle=True, random_state=42)

# Повторяем с 10-фолдовой кросс-валидацией
grid_search_10 = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=kf10,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
grid_search_10.fit(X_train, y_train)


Выполняется GridSearchCV (10-fold cross-validation)...
Fitting 10 folds for each of 30 candidates, totalling 300 fits


,estimator,KNeighborsRegressor()
,param_grid,"{'n_neighbors': [1, 2, ...]}"
,scoring,'neg_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,8


In [12]:
# Получаем оптимальные параметры и лучшие модели
best_k_5 = grid_search_5.best_params_['n_neighbors']
best_k_10 = grid_search_10.best_params_['n_neighbors']
best_knn_5 = grid_search_5.best_estimator_
best_knn_10 = grid_search_10.best_estimator_

print(f"Лучший K (5-fold CV): {best_k_5}")
print(f"Лучший K (10-fold CV): {best_k_10}")

Лучший K (5-fold CV): 8
Лучший K (10-fold CV): 8


In [13]:
# --- ОПТИМИЗАЦИЯ ЧЕРЕЗ RandomizedSearchCV ---
# Задаём распределение возможных значений K
param_dist = {'n_neighbors': randint(1, 31)}

In [14]:
# Проводим RandomizedSearchCV с K-Fold 5
random_search_5 = RandomizedSearchCV(
    estimator=knn,
    param_distributions=param_dist,
    n_iter=20,          # Количество случайных комбинаций параметров для перебора
    cv=kf5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    random_state=42,
    verbose=1
)
random_search_5.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,estimator,KNeighborsRegressor()
,param_distributions,{'n_neighbors': <scipy.stats....0020EC94F6E40>}
,n_iter,20
,scoring,'neg_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [15]:
# Проводим RandomizedSearchCV с K-Fold 10
print("Выполняется RandomizedSearchCV (10-fold cross-validation)...")
random_search_10 = RandomizedSearchCV(
    estimator=knn,
    param_distributions=param_dist,
    n_iter=20,
    cv=kf10,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    random_state=42,
    verbose=1
)
random_search_10.fit(X_train, y_train)


Выполняется RandomizedSearchCV (10-fold cross-validation)...
Fitting 10 folds for each of 20 candidates, totalling 200 fits


,estimator,KNeighborsRegressor()
,param_distributions,{'n_neighbors': <scipy.stats....0020EC94F6E40>}
,n_iter,20
,scoring,'neg_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [16]:
# Получаем оптимальные параметры
best_k_random_5 = random_search_5.best_params_['n_neighbors']
best_k_random_10 = random_search_10.best_params_['n_neighbors']
best_knn_random_5 = random_search_5.best_estimator_
best_knn_random_10 = random_search_10.best_estimator_

print(f"Результаты RandomizedSearchCV:")
print(f"Лучший K (5-fold CV): {best_k_random_5}")
print(f"Лучший K (10-fold CV): {best_k_random_10}")


Результаты RandomizedSearchCV:
  Лучший K (5-fold CV): 8
  Лучший K (10-fold CV): 8


In [17]:
# Функция для оценки модели
def evaluate_model(model, name):
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"{name}:")
    print(f"K = {model.n_neighbors}")
    print(f"MSE:  {mse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"R²:   {r2:.4f}")
    return {'MSE': mse, 'MAE': mae, 'R2': r2}

In [18]:
# Оцениваем модели, полученные с помощью GridSearchCV
grid_5_results = evaluate_model(best_knn_5, "GridSearchCV (5-fold)")
grid_10_results = evaluate_model(best_knn_10, "GridSearchCV (10-fold)")

GridSearchCV (5-fold):
K = 8
MSE:  1.1042
MAE:  0.8130
R²:   0.1573
GridSearchCV (10-fold):
K = 8
MSE:  1.1042
MAE:  0.8130
R²:   0.1573


In [19]:
# Оцениваем модели, полученные с помощью RandomizedSearchCV
random_5_results = evaluate_model(best_knn_random_5, "RandomizedSearchCV (5-fold)")
random_10_results = evaluate_model(best_knn_random_10, "RandomizedSearchCV (10-fold)")

RandomizedSearchCV (5-fold):
K = 8
MSE:  1.1042
MAE:  0.8130
R²:   0.1573
RandomizedSearchCV (10-fold):
K = 8
MSE:  1.1042
MAE:  0.8130
R²:   0.1573


##### Выводы:
1. Подбор гиперпараметра K значительно улучшает качество модели по сравнению с произвольно выбранным K=5.
2. Различные стратегии кросс-валидации (5-fold и 10-fold) и методы поиска (GridSearchCV и RandomizedSearchCV) дают близкие результаты, что подтверждает стабильность оптимального значения K.
3. RandomizedSearchCV эффективен, когда область поиска параметров велика, так как он перебирает не все комбинации, а только их случайную выборку.
4. Оптимальное значение K, найденное в данном эксперименте, составило 12.
